# Probabilistic Calibration via Conformalized Quantile Regression (Week 5, Task 3)

## PRE-REGISTRATION (declared before results)

### Goal
Build a probabilistic 30-day-ahead forecast for corn returns using quantile regression 
(q10, q50, q90) + conformal calibration to achieve 80% empirical coverage. 
Evaluate coverage OVERALL, per REGIME, and per YEAR. Use vanilla features only 
(no regime drivers, as regime features did not lift point accuracy in Task 2).

### Feature Set (FROZEN — VANILLA ONLY)
- ret_1, ret_5, ret_10, ret_21, ret_63 (lagged log returns)
- vol_21 (21-day rolling realized volatility)
- sin_doy, cos_doy (seasonal day-of-year indicators)
Total: 8 features

### Quantile Regression Setup
- **Model**: XGBoost with `reg:quantileerror` loss
- **Quantiles**: q10, q50 (median), q90
- **Implementation**: 
  - If XGBoost >= 2.0 supports multi-quantile in one call, use it
  - Otherwise fit three separate models (one per quantile)
- **Constraint**: Enforce monotonic sort post-prediction: q10 ≤ q50 ≤ q90 per row (no quantile crossing)
- **XGBoost hyperparameters** (IDENTICAL to Week-4 vanilla baseline):
  - n_estimators=300, max_depth=3, learning_rate=0.03
  - subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, random_state=0

### Conformal Quantile Regression (CQR) Setup
- **Calibration target**: 80% empirical coverage (alpha = 0.2)
- **Calibration data**: Recent trailing ~1 year (~252 trading days) of in-fold training data
- **Tool**: mapie's CQR class (version-aware)
- **Decision gate** (pre-registered): 
  1. **Marginal CQR first**: Single calibration, single correction Q
  2. Measure per-regime coverage
  3. **Promote-to-conditional rule**:
     - If all regimes within ~70-90% coverage → **MARGINAL STANDS** (ship it)
     - If any regime under-covers (< 70%, esp. high-vol regimes) → **PROMOTE to VOL-TERCILE CONDITIONAL**:
       Bucket by vol_21 into 3 terciles, apply separate CQR per tercile
  4. Document choice + numbers

### Walk-Forward Evaluation
- **Folds**: Reuse Task-2 harness (expanding window, step=21, embargo=30, OOS 2020-2026)
- **Per fold**:
  1. Split in-fold data: older chunk → model-train, recent ~252 days → conformal-calibration
  2. Fit q10/q50/q90 quantile models on train
  3. Conformalize on calib
  4. Predict intervals on test
- **Output**: OOS results with [low, high, realized, covered, width] per point

### Coverage Measurement (THE DELIVERABLE)
For every OOS test point, record:
- Predicted interval [q10, q90] (after CQR adjustment)
- Realized outcome
- Covered = (outcome ∈ [q10, q90])
- Width = q90 - q10

Then compute + report:
- **Empirical coverage OVERALL** (% of OOS points covered)
- **Per-regime coverage** (group OOS by the seven PELT regime labels)
- **Per-year coverage** (group by calendar year)
- **Mean width OVERALL and per-regime** (coverage + width reported together)

Target: 80% coverage. Width should be reasonable (not bloated by over-correction).

### Final Production Model
Fit quantile models + CQR on ALL data through latest date, serialize to joblib:
- `quantile_models.joblib`: (q10_model, q50_model, q90_model)
- `conformal_calibration.joblib`: CQR calibration object
These are committed to the repo for Render to load at startup.

### Transform Note
The model predicts 30-day log RETURN quantiles. The live API contract expects PRICE quantiles.
Transformation (documented, not implemented here):
  `price_low = last_price * exp(q10)`
  `price_point = last_price * exp(q50)`
  `price_high = last_price * exp(q90)`

In [ ]:
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import xgboost as xgb
import joblib
from pathlib import Path

# mapie import (version-aware)
try:
    from mapie.regression import ConformalizedQuantileRegressor
    MAPIE_V1 = True
    print("✓ mapie v1.x detected")
except ImportError:
    from mapie.quantile_regression import QuantileRegressor
    MAPIE_V1 = False
    print("✓ mapie v0.x detected")

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print(f"Root: {ROOT}")

: 

In [ ]:
# Load panel from DuckDB
con = duckdb.connect(str(ROOT / "data" / "openag.duckdb"), read_only=True)
panel = con.sql('SELECT * FROM panel ORDER BY "Date"').df()
con.close()

panel["Date"] = pd.to_datetime(panel["Date"]).astype("datetime64[ns]")
panel = panel.set_index("Date")

print(f"Panel: {panel.shape[0]} rows, {panel.index.min().date()} → {panel.index.max().date()}")
print(f"Columns: {list(panel.columns)}")
print(f"\nRegime distribution:")
print(panel["regime"].value_counts().sort_index())

In [ ]:
H = 30  # forecast horizon

df = panel.copy()

# Target: 30-day-ahead log return
df["y"] = np.log(df["Close"].shift(-H)) - np.log(df["Close"])

# Vanilla features (unchanged from Week 4)
for lag in [1, 5, 10, 21, 63]:
    df[f"ret_{lag}"] = np.log(df["Close"]) - np.log(df["Close"].shift(lag))

df["vol_21"] = df["rolling_vol"]

doy = df.index.dayofyear
df["sin_doy"] = np.sin(2 * np.pi * doy / 365.25)
df["cos_doy"] = np.cos(2 * np.pi * doy / 365.25)

FEATURES_VANILLA = [
    "ret_1", "ret_5", "ret_10", "ret_21", "ret_63",
    "vol_21", "sin_doy", "cos_doy"
]

print(f"Vanilla features: {len(FEATURES_VANILLA)}")
print(f"Target (y) non-NaN: {df['y'].notna().sum()}")

# Coverage: rows with no NaN in vanilla features or target
model_df = df.dropna(subset=FEATURES_VANILLA + ["y"])
print(f"Model-ready rows: {len(model_df)}")


In [ ]:
def walk_forward_folds(idx, oos_start, step=21, embargo=H, min_train=250):
    """Expanding-window folds; train target windows never reach test region (embargo=H)."""
    idx = pd.DatetimeIndex(idx)
    start = int(np.searchsorted(idx, pd.Timestamp(oos_start)))
    folds = []
    for origin in range(start, len(idx), step):
        train_pos = np.arange(0, origin - embargo)
        test_pos = np.arange(origin, min(origin + step, len(idx)))
        if len(train_pos) >= min_train and len(test_pos) > 0:
            folds.append((train_pos, test_pos))
    return folds


def evaluate(folds, data, features, fit_predict_fn):
    """Run fit_predict_fn(train_df, test_df, features) -> yhat across folds; stack OOS predictions."""
    parts = []
    for train_pos, test_pos in folds:
        tr, te = data.iloc[train_pos], data.iloc[test_pos]
        result = fit_predict_fn(tr, te, features)
        # result can be a DataFrame (with q10/q50/q90 columns) or array (point forecast)
        if isinstance(result, pd.DataFrame):
            pred_df = result.copy()
        else:
            pred_df = pd.DataFrame({"yhat": np.asarray(result)}, index=te.index)
        
        # Add true targets
        pred_df["y"] = te["y"].to_numpy()
        parts.append(pred_df)
    
    return pd.concat(parts)


# Create folds (identical to Task 2)
folds = walk_forward_folds(model_df.index, oos_start="2020-01-01")
print(f"Folds: {len(folds)} expanding windows")
print(f"OOS period: {model_df.index[folds[0][1][0]].date()} → {model_df.index[-1].date()}")


In [ ]:
def enforce_quantile_monotonicity(q_low, q_mid, q_high):
    """Ensure q_low <= q_mid <= q_high via sorting per row."""
    quantiles = np.column_stack([q_low, q_mid, q_high])
    sorted_q = np.sort(quantiles, axis=1)
    return sorted_q[:, 0], sorted_q[:, 1], sorted_q[:, 2]


# Test: verify the function works
test_q10 = np.array([0.01, -0.02, 0.00])
test_q50 = np.array([-0.01, 0.00, 0.01])
test_q90 = np.array([0.05, 0.02, 0.03])
q10_s, q50_s, q90_s = enforce_quantile_monotonicity(test_q10, test_q50, test_q90)
print(f"Monotonic sort test:")
print(f"  Input:  q10={test_q10}, q50={test_q50}, q90={test_q90}")
print(f"  Output: q10={q10_s}, q50={q50_s}, q90={q90_s}")
assert np.all(q10_s <= q50_s) and np.all(q50_s <= q90_s), "Monotonicity enforced ✓"


In [ ]:
import xgboost as xgb

print(f"XGBoost version: {xgb.__version__}")

# Check if multi-quantile in one model is supported (XGBoost >= 2.0)
# For now, use three separate models for robustness across versions
MULTI_QUANTILE_SUPPORTED = tuple(map(int, xgb.__version__.split('.')[:2])) >= (2, 0)
print(f"Multi-quantile in one call: {MULTI_QUANTILE_SUPPORTED}")

XGB_PARAMS = dict(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=0,
    n_jobs=-1,
)
print(f"XGB params (shared across quantiles): {XGB_PARAMS}")


In [ ]:
def fit_predict_quantiles(tr, te, features, quantiles=[0.1, 0.5, 0.9]):
    """Fit separate XGBoost quantile regressors per quantile; predict on test."""
    q_preds = {}
    
    for q in quantiles:
        model = xgb.XGBRegressor(
            objective="reg:quantileerror",
            quantile_alpha=q,
            **XGB_PARAMS
        )
        model.fit(tr[features], tr["y"])
        q_preds[q] = model.predict(te[features])
    
    q10 = q_preds[0.1]
    q50 = q_preds[0.5]
    q90 = q_preds[0.9]
    
    # Enforce monotonic sort
    q10, q50, q90 = enforce_quantile_monotonicity(q10, q50, q90)
    
    return pd.DataFrame(
        {
            "q10": q10,
            "q50": q50,
            "q90": q90,
        },
        index=te.index,
    )


# Test on a small fold
test_fold_idx = 0
tr_pos, te_pos = folds[test_fold_idx]
tr, te = model_df.iloc[tr_pos], model_df.iloc[te_pos]

print(f"Test fold {test_fold_idx}:")
print(f"  Train: {len(tr)} rows, {tr.index.min().date()} → {tr.index.max().date()}")
print(f"  Test:  {len(te)} rows, {te.index.min().date()} → {te.index.max().date()}")

test_result = fit_predict_quantiles(tr, te, FEATURES_VANILLA)
print(f"\nQuantile predictions (first 5 rows):")
print(test_result.head())
print(f"\nMonotonicity check:")
print(f"  q10 <= q50: {(test_result['q10'] <= test_result['q50']).all()}")
print(f"  q50 <= q90: {(test_result['q50'] <= test_result['q90']).all()}")


In [ ]:
def fit_predict_with_cqr(tr, te, features, alpha=0.2, calib_size=252):
    """
    Split tr into model-train (older) + calib (recent ~1yr), fit quantiles,
    conformalize on calib, predict intervals on te.
    """
    # Split training data: keep older for model training, recent for CQR calibration
    calib_start_idx = max(0, len(tr) - calib_size)
    tr_model = tr.iloc[:calib_start_idx]
    tr_calib = tr.iloc[calib_start_idx:]
    
    # Fit quantile models on tr_model
    models = {}
    for q in [0.1, 0.5, 0.9]:
        model = xgb.XGBRegressor(
            objective="reg:quantileerror",
            quantile_alpha=q,
            **XGB_PARAMS
        )
        model.fit(tr_model[features], tr_model["y"])
        models[q] = model
    
    # Predict on calibration set
    q10_calib = models[0.1].predict(tr_calib[features])
    q50_calib = models[0.5].predict(tr_calib[features])
    q90_calib = models[0.9].predict(tr_calib[features])
    
    # Enforce monotonicity on calib
    q10_calib, q50_calib, q90_calib = enforce_quantile_monotonicity(q10_calib, q50_calib, q90_calib)
    
    # Prepare X and y for CQR
    X_calib = tr_calib[features].values
    y_calib = tr_calib["y"].values
    
    # Use q50 (median) as the base predictor for CQR
    y_pred_calib = q50_calib
    
    # Compute residuals for calibration
    lower_calib = q10_calib
    upper_calib = q90_calib
    
    # Predict on test set
    q10_test = models[0.1].predict(te[features])
    q50_test = models[0.5].predict(te[features])
    q90_test = models[0.9].predict(te[features])
    
    q10_test, q50_test, q90_test = enforce_quantile_monotonicity(q10_test, q50_test, q90_test)
    
    # Apply CQR correction (simple approach: expand intervals by a learned factor)
    # Compute quantile-based correction from calibration
    correction_lower = np.quantile(lower_calib - y_calib, alpha / 2)
    correction_upper = np.quantile(upper_calib - y_calib, 1 - alpha / 2)
    
    q10_corrected = q10_test + correction_lower
    q90_corrected = q90_test + correction_upper
    
    return pd.DataFrame(
        {
            "q10": q10_corrected,
            "q50": q50_test,
            "q90": q90_corrected,
        },
        index=te.index,
    )


# Run walk-forward evaluation with CQR
print("Running walk-forward evaluation with marginal CQR...")
print("(This may take 10-15 minutes for 76 folds)")

oos_results = []
for i, (tr_pos, te_pos) in enumerate(folds):
    if i % 10 == 0:
        print(f"  Fold {i}/{len(folds)}...")
    
    tr, te = model_df.iloc[tr_pos], model_df.iloc[te_pos]
    pred_df = fit_predict_with_cqr(tr, te, FEATURES_VANILLA)
    pred_df["y"] = te["y"].values
    pred_df["regime"] = te["regime"].values
    
    oos_results.append(pred_df)

oos = pd.concat(oos_results)
print(f"\nOOS results: {len(oos)} points")
print(oos.head(10))

In [ ]:
def compute_coverage(oos):
    """Compute empirical coverage: overall, per-regime, per-year."""
    oos["covered"] = (oos["y"] >= oos["q10"]) & (oos["y"] <= oos["q90"])
    oos["width"] = oos["q90"] - oos["q10"]
    
    # Overall
    overall_coverage = oos["covered"].mean()
    overall_width = oos["width"].mean()
    
    # Per regime
    regime_coverage = oos.groupby("regime", observed=True)["covered"].agg(["mean", "count"])
    regime_coverage.columns = ["coverage", "count"]
    regime_width = oos.groupby("regime", observed=True)["width"].mean()
    regime_stats = pd.concat([regime_coverage, regime_width.rename("width")], axis=1)
    
    # Per year
    oos["year"] = oos.index.year
    year_coverage = oos.groupby("year", observed=True)["covered"].agg(["mean", "count"])
    year_coverage.columns = ["coverage", "count"]
    year_width = oos.groupby("year", observed=True)["width"].mean()
    year_stats = pd.concat([year_coverage, year_width.rename("width")], axis=1)
    
    return {
        "overall_coverage": overall_coverage,
        "overall_width": overall_width,
        "regime_stats": regime_stats,
        "year_stats": year_stats,
    }


marginal_coverage = compute_coverage(oos)

print("=" * 80)
print("MARGINAL CQR COVERAGE (OVERALL)")
print("=" * 80)
print(f"Coverage: {marginal_coverage['overall_coverage']:.2%} (target: 80.00%)")
print(f"Width:    {marginal_coverage['overall_width']:.4f}")
print()

print("=" * 80)
print("COVERAGE BY REGIME")
print("=" * 80)
regime_stats = marginal_coverage["regime_stats"].copy()
regime_stats["coverage"] = regime_stats["coverage"] * 100
print(regime_stats.to_string())
print()

print("=" * 80)
print("COVERAGE BY YEAR")
print("=" * 80)
year_stats = marginal_coverage["year_stats"].copy()
year_stats["coverage"] = year_stats["coverage"] * 100
print(year_stats.to_string())

## DECISION GATE: Marginal vs. Conditional CQR

### Rule (declared BEFORE examining results):
1. **Measure per-regime coverage** from marginal CQR above
2. **Check coverage balance**:
   - If **all regimes within 70–90%**: Marginal CQR stands (ship it)
   - If **any regime < 70%** (esp. high-vol regimes like burst_spr21/burst_spr23): Promote to VOL-TERCILE CONDITIONAL
3. **Conditional approach** (if needed):
   - Bucket OOS test by vol_21 into 3 terciles (low, mid, high volatility)
   - Apply separate CQR calibration + correction per tercile
   - Target: achieve ~80% coverage within each tercile

### Rationale
- Thin buckets (7 regimes) lead to unstable calibration + bloated intervals
- Vol terciles (3 buckets) are coarse, stable, and have causal grounding
- High-vol regimes need wider intervals to achieve target coverage

In [ ]:
def fit_predict_with_vol_tercile_cqr(tr, te, features, alpha=0.2, calib_size=252):
    """
    CQR with conditioning on vol_21 terciles.
    """
    # Split training data
    calib_start_idx = max(0, len(tr) - calib_size)
    tr_model = tr.iloc[:calib_start_idx]
    tr_calib = tr.iloc[calib_start_idx:]
    
    # Fit quantile models on tr_model (same as before)
    models = {}
    for q in [0.1, 0.5, 0.9]:
        model = xgb.XGBRegressor(
            objective="reg:quantileerror",
            quantile_alpha=q,
            **XGB_PARAMS
        )
        model.fit(tr_model[features], tr_model["y"])
        models[q] = model
    
    # Predict on test set
    q10_test = models[0.1].predict(te[features])
    q50_test = models[0.5].predict(te[features])
    q90_test = models[0.9].predict(te[features])
    
    q10_test, q50_test, q90_test = enforce_quantile_monotonicity(q10_test, q50_test, q90_test)
    
    # Compute vol tercile boundaries using tr_calib
    vol_terciles = np.percentile(tr_calib["vol_21"], [33.33, 66.67])
    
    # Assign test points to terciles
    te_vol_tercile = np.digitize(te["vol_21"], vol_terciles)  # 0, 1, 2
    
    # Apply per-tercile CQR corrections
    q10_corrected = q10_test.copy()
    q90_corrected = q90_test.copy()
    
    for tercile in range(3):
        mask = te_vol_tercile == tercile
        if not mask.any():
            continue
        
        # Get calib data for this tercile
        calib_vol_tercile = np.digitize(tr_calib["vol_21"], vol_terciles)
        calib_mask = calib_vol_tercile == tercile
        
        if calib_mask.sum() < 20:
            continue  # Skip thin terciles
        
        y_calib_t = tr_calib.iloc[calib_mask]["y"].values
        q10_calib_t = models[0.1].predict(tr_calib.iloc[calib_mask][features])
        q90_calib_t = models[0.9].predict(tr_calib.iloc[calib_mask][features])
        
        q10_calib_t, _, q90_calib_t = enforce_quantile_monotonicity(q10_calib_t, q10_calib_t, q90_calib_t)
        
        # Quantile-based corrections per tercile
        correction_lower = np.quantile(q10_calib_t - y_calib_t, alpha / 2)
        correction_upper = np.quantile(q90_calib_t - y_calib_t, 1 - alpha / 2)
        
        q10_corrected[mask] = q10_test[mask] + correction_lower
        q90_corrected[mask] = q90_test[mask] + correction_upper
    
    return pd.DataFrame(
        {
            "q10": q10_corrected,
            "q50": q50_test,
            "q90": q90_corrected,
            "vol_tercile": te_vol_tercile,
        },
        index=te.index,
    )


# Decision: evaluate marginal coverage and decide
print("\n" + "=" * 80)
print("DECISION GATE")
print("=" * 80)

regime_cov = marginal_coverage["regime_stats"]["coverage"]
under_coverage = (regime_cov < 0.70).any()

if not under_coverage:
    print("✓ All regimes within 70–90% coverage")
    print("  → MARGINAL CQR STANDS (ship it)")
    USE_CONDITIONAL = False
else:
    bad_regimes = regime_cov[regime_cov < 0.70]
    print(f"✗ Regimes under-covering (< 70%):")
    for regime, cov in bad_regimes.items():
        print(f"    {regime}: {cov:.1%}")
    print("  → PROMOTE to VOL-TERCILE CONDITIONAL CQR")
    USE_CONDITIONAL = True

print()

In [ ]:
if USE_CONDITIONAL:
    print("Running walk-forward evaluation with VOL-TERCILE CONDITIONAL CQR...")
    print("(This may take 10-15 minutes)")
    
    oos_results_cond = []
    for i, (tr_pos, te_pos) in enumerate(folds):
        if i % 10 == 0:
            print(f"  Fold {i}/{len(folds)}...")
        
        tr, te = model_df.iloc[tr_pos], model_df.iloc[te_pos]
        pred_df = fit_predict_with_vol_tercile_cqr(tr, te, FEATURES_VANILLA)
        pred_df["y"] = te["y"].values
        pred_df["regime"] = te["regime"].values
        
        oos_results_cond.append(pred_df)
    
    oos = pd.concat(oos_results_cond)
    cqr_coverage = compute_coverage(oos)
    
    print(f"\nVOL-TERCILE CONDITIONAL CQR:")
    print(f"Coverage: {cqr_coverage['overall_coverage']:.2%} (target: 80.00%)")
    print(f"Width:    {cqr_coverage['overall_width']:.4f}")
    print()
    
    print("Coverage by regime:")
    regime_stats_cond = cqr_coverage["regime_stats"].copy()
    regime_stats_cond["coverage"] = regime_stats_cond["coverage"] * 100
    print(regime_stats_cond.to_string())
    
    FINAL_COVERAGE = cqr_coverage
else:
    print("Using MARGINAL CQR (no conditioning needed)")
    FINAL_COVERAGE = marginal_coverage

print()

## Decision Summary

**Marginal CQR performance:**
- Overall coverage: [FILL IN: XX.X%]
- Best regime: [FILL IN: regime_name at XX.X%]
- Worst regime: [FILL IN: regime_name at XX.X%]

**Conditional CQR (if applied) performance:**
- Overall coverage: [FILL IN: XX.X%]
- Improvement: [FILL IN: +X.X% or "not needed"]

**Final choice:** [MARGINAL / CONDITIONAL]

**Rationale:** [1-2 sentences explaining why this choice optimizes for calibrated 80% coverage]

---

### Exchangeability Caveat
Conformal prediction guarantees assume exchangeability (i.i.d. data). Time series breaks this assumption.
Therefore, the empirical coverage measurement (not the conformal theorem) is what we trust. 
The coverage numbers above are empirical validation on the OOS test set.